### Chapter 3.7 - Weight Decay

Weight decay is a regularization method that discourages large weights. It helps control overfitting by adding a size penalty to the training objective.


In [ ]:
import math
import random
import numpy as np
import torch


### 3.7.1 Norms and Weight Decay

#### 1. Intuition

A norm measures the size of a vector. The L2 norm squares each value, sums the squares, and takes a square root.

Weight decay penalizes large weights by adding a term based on weight size to the loss.

#### 2. Why this exists

Large weights can make a model overly sensitive to small input changes. Penalizing weight size encourages simpler functions.


#### 3. Examples

Manual L2 penalty for a small weight vector.


In [ ]:
w = [3.0, 4.0]
l2_squared = sum(value * value for value in w)
penalty = 0.5 * l2_squared
penalty


PyTorch L2 penalty.


In [ ]:
w = torch.tensor([3.0, 4.0])
penalty = 0.5 * torch.sum(w ** 2)
penalty


#### 4. Step-by-step breakdown

The manual code squares each weight and sums the results.

The factor `0.5` is a convenience because it simplifies derivatives.

In PyTorch, `w ** 2` squares each weight elementwise.

`torch.sum(...)` reduces all squared values into one scalar penalty.

#### 5. Connection to ML systems

Weight decay modifies the training objective from `data loss` to `data loss plus weight-size penalty`. The model must fit data while keeping weights controlled.

#### 6. Common confusion points

- Weight decay usually penalizes weights, not the bias.
- The penalty is added during training, not after training.
- A larger penalty strength forces smaller weights more strongly.
- Smaller weights do not automatically mean better predictions.


### 3.7.2 High-Dimensional Linear Regression

#### 1. Intuition

High-dimensional data has many features. If the number of features is large compared with the number of examples, overfitting becomes easier.

The model has many ways to fit training noise when it has many weights.

#### 2. Why this exists

Weight decay is especially useful when the model has enough flexibility to memorize the training data.


#### 3. Examples

Create more features than examples.


In [ ]:
torch.manual_seed(0)
num_train = 5
num_features = 20
X = torch.randn(num_train, num_features)
true_w = torch.zeros(num_features)
true_w[:2] = torch.tensor([2.0, -3.4])
y = X @ true_w + 0.01 * torch.randn(num_train)
X.shape, y.shape


#### 4. Step-by-step breakdown

`num_features = 20` means each example has 20 input values.

`num_train = 5` means there are only 5 training examples.

Only the first two true weights are nonzero.

The model does not know this sparse structure unless we guide it with assumptions or regularization.

#### 5. Connection to ML systems

Real high-dimensional problems include text features, genomics, recommendation systems, and wide tabular data. Regularization helps reduce reliance on accidental patterns.

#### 6. Common confusion points

- Many features increase the chance of fitting noise.
- More dimensions are not automatically bad, but they require enough data or regularization.
- True useful features may be sparse.
- Synthetic high-dimensional data is useful for testing overfitting behavior.


### 3.7.3 Implementation from Scratch

#### 1. Intuition

From scratch, weight decay is implemented by adding an L2 penalty to the loss before calling `backward()`.

The penalty strength is usually written as lambda. In code, use a name like `wd` because `lambda` is a Python keyword.

#### 2. Why this exists

Adding the penalty manually shows exactly how regularization changes the objective and therefore the gradients.


#### 3. Examples

Define the penalty and add it to a data loss.


In [ ]:
def l2_penalty(w):
    return 0.5 * torch.sum(w ** 2)

w = torch.randn(20, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
y_hat = X @ w + b
data_loss = ((y_hat - y) ** 2).mean()
loss = data_loss + 0.1 * l2_penalty(w)
loss


Compute gradients from the regularized objective.


In [ ]:
loss.backward()
w.grad.shape, b.grad.shape


#### 4. Step-by-step breakdown

`l2_penalty(w)` computes the size penalty for the weight vector.

`data_loss` measures prediction error.

`0.1 * l2_penalty(w)` scales the regularization strength.

The final `loss` combines prediction error and weight penalty.

Calling `backward()` computes gradients for the combined objective.

#### 5. Connection to ML systems

This is the transparent version of weight decay. Later optimizer options can apply the same idea more concisely.

#### 6. Common confusion points

- Penalize `w`, not usually `b`.
- The regularization coefficient controls penalty strength.
- The penalty changes gradients, not just reported loss.
- Clear gradients before another backward pass in a real training loop.


### 3.7.4 Concise Implementation

#### 1. Intuition

PyTorch optimizers can apply weight decay directly through an optimizer setting.

This keeps the loss expression focused on prediction error while the optimizer handles the weight penalty.

#### 2. Why this exists

The concise option reduces boilerplate and matches common PyTorch practice.


#### 3. Examples

Create an optimizer with weight decay.


In [ ]:
net = torch.nn.Linear(20, 1)
optimizer = torch.optim.SGD(net.parameters(), lr=0.03, weight_decay=0.1)
loss_fn = torch.nn.MSELoss()
y_col = y.reshape(-1, 1)
optimizer.zero_grad()
loss = loss_fn(net(X), y_col)
loss.backward()
optimizer.step()
loss


#### 4. Step-by-step breakdown

`torch.nn.Linear(20, 1)` creates a linear model for 20 features.

`weight_decay=0.1` tells the optimizer to include weight decay in parameter updates.

The explicit `loss` is only prediction loss here.

`optimizer.step()` applies the update using gradients and the optimizer's weight-decay rule.

#### 5. Connection to ML systems

Most concise PyTorch implementations pass `weight_decay` to the optimizer. Some advanced optimizers handle weight decay slightly differently, so optimizer documentation matters.

#### 6. Common confusion points

- Built-in weight decay is an optimizer setting, not a separate visible loss term.
- Check whether the optimizer applies weight decay to all parameters or selected groups.
- Bias parameters are often excluded in larger systems.
- Concise code still follows zero-grad, loss, backward, step.


### 3.7.5 Summary

#### 1. Intuition

Weight decay is regularization that discourages large weights.

It can be written manually as an L2 penalty or requested through optimizer settings.

#### 2. Why this exists

The goal is better generalization, especially when the model can overfit training noise.


#### 3. Examples

Compare the two implementation styles.


In [ ]:
styles = {
    "scratch": "add wd * l2_penalty(w) to loss",
    "concise": "pass weight_decay to optimizer",
}
styles


#### 4. Step-by-step breakdown

The scratch version makes the penalty visible in the loss.

The concise version moves the penalty into the optimizer.

Both are attempts to control model complexity by discouraging large weights.

#### 5. Connection to ML systems

Weight decay is one of the first regularization techniques used in deep learning systems. Later chapters add more regularization tools.

#### 6. Common confusion points

- Regularization improves expected behavior; it does not guarantee better validation loss every time.
- Too much weight decay can cause underfitting.
- Weight decay strength is a hyperparameter.
- Always compare training and validation performance.


### 3.7.6 Exercises

#### 1. Intuition

These exercises practice adding and interpreting weight penalties.

#### 2. Why this exists

The goal is to connect the regularization coefficient, weight size, and objective value.


#### 3. Examples

Exercise 1: compute L2 penalty for a different weight vector.


In [ ]:
w_test = torch.tensor([1.0, -2.0, 2.0])
l2_penalty(w_test)


Exercise 2: compare regularized losses with two penalty strengths.


In [ ]:
data_loss = torch.tensor(1.5)
penalty = l2_penalty(w_test)
loss_small = data_loss + 0.01 * penalty
loss_large = data_loss + 1.0 * penalty
loss_small, loss_large


#### 4. Step-by-step breakdown

Exercise 1 checks the L2 penalty formula.

Exercise 2 checks that larger regularization strength increases the objective more for the same weights.

The model would therefore feel more pressure to reduce weight size.

#### 5. Connection to ML systems

This prepares you to tune weight decay using validation performance rather than guessing.

#### 6. Common confusion points

- A larger penalty coefficient does not always mean better generalization.
- The same coefficient can behave differently with different loss scales.
- Regularization changes training dynamics.
- Tune weight decay on validation data, not test data.
